# Lab 3 — Coupon Collector

**확률통계 · Week 3 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. "전부 모으려면 몇 번?"을 **시뮬레이션으로** 답한다.
2. **기댓값의 선형성**으로 같은 답을 **수식으로** 얻고 둘을 비교한다.
3. 평균만으로는 부족하다는 것 — **분산**을 눈으로 확인한다.

⏱ **예상 소요 시간: 35분**

---

### 문제

과자 상자마다 스티커가 **1장** 무작위로 들어 있다. 종류는 $n$ 가지, 전부 같은 확률.

> **$n$ 종류를 모두 모으려면 평균 몇 상자를 사야 할까?**

$n = 6$ 이면 감이 오는가? 6상자? 10상자? 먼저 **직접 추측해보자.**

**나의 추측 (n=6):** (여기에 작성)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260302)
print("준비 완료")

## Part 1. 한 번 모아보기

규칙을 그대로 코드로 옮긴다. **다 모을 때까지** 계속 뽑는다.

### 실습 1 — 수집 한 판

In [ ]:
def collect_once(n, rng):
    seen = np.zeros(n, dtype=bool)   # 각 종류를 봤는지 기록
    count = 0
    while not seen.all():            # 전부 True가 될 때까지
        k = rng.integers(0, n)       # 무작위로 한 장
        seen[k] = True               # 봤다고 기록
        # TODO 1: 뽑은 횟수를 1 늘리세요.  힌트: count += 1
    return count


print("n=6 한 번 해보기:", collect_once(6, rng))
print("n=6 또 해보기  :", collect_once(6, rng))

> TODO 1을 채우기 전에는 결과가 계속 0으로 나온다. 에러는 나지 않으니 당황하지 말 것.

두 번 돌린 결과가 다르다. **매번 다르다.** 그래서 여러 번 해서 평균을 내야 한다.

### 실습 2 — 평균 구하기

In [ ]:
def collect_mean(n, repeats=2000, seed=20260302):
    rng = np.random.default_rng(seed)
    draws = [collect_once(n, rng) for _ in range(repeats)]
    # TODO 2: draws의 평균과 draws 배열을 함께 돌려주세요
    return 0.0, np.array(draws)


mean6, draws6 = collect_mean(6)
print(f"n=6 평균: {mean6:.2f} 상자")

## Part 2. 수식으로 풀기 — 선형성의 힘

전체 횟수 $T$ 의 분포를 구하는 것은 어렵다. 하지만 **쪼개면** 쉽다.

- $T_1$ = 첫 번째 새 스티커를 얻기까지 (당연히 1번)
- $T_2$ = 그다음 새 스티커를 얻기까지
- ...
- $T_n$ = 마지막 하나를 얻기까지

$$T = T_1 + T_2 + \cdots + T_n$$

이미 $k-1$ 종류를 모았다면, 새것이 나올 확률은 $\frac{n-(k-1)}{n}$ 이다.
"확률 $p$ 인 일이 처음 성공할 때까지의 평균 횟수는 $1/p$" 이므로 (4주차 Geometric에서 증명한다)

$$\mathbb{E}[T_k] = \frac{n}{n-k+1}
\quad\Longrightarrow\quad
\mathbb{E}[T] = \sum_{k=1}^{n} \frac{n}{n-k+1} = n\left(1 + \frac{1}{2} + \cdots + \frac{1}{n}\right) = n H_n$$

> **$T_k$ 들은 서로 독립이 아니어도 된다.** 선형성은 그것을 묻지 않는다.

### 실습 3 — 이론값 계산

In [ ]:
def theory(n):
    # TODO 3: 조화수 H_n = 1 + 1/2 + ... + 1/n 을 구하고 n을 곱해 돌려주세요
    harmonic = 0.0
    return n * harmonic


for n in [6, 10, 50]:
    print(f"n={n:>3} 이론값 {theory(n):8.2f}")

### 실습 4 — 시뮬레이션과 이론값 비교

$n = 2 \ldots 30$ 에 대해 두 값을 겹쳐 그린다.

> 📌 축 이름과 제목은 **영어로**. (Colab에는 한글 폰트가 없다)

In [ ]:
ns = range(2, 31)
sim = [collect_mean(n, repeats=500)[0] for n in ns]
theo = [theory(n) for n in ns]

plt.figure(figsize=(7, 4))
# TODO 4: 시뮬레이션 결과는 점으로, 이론값은 선으로 그리세요
#         힌트: plt.plot(list(ns), sim, "o", label="simulation")

plt.xlabel("Number of coupon types n")
plt.ylabel("Mean draws to collect all")
plt.title("Coupon collector")
plt.legend()
plt.show()

## Part 3. 평균만으로는 부족하다

$n=6$ 의 평균은 약 **14.7상자**다. 그러면 **15상자만 사면 되는가?**

### 실습 5 — 분포를 보자

In [ ]:
plt.figure(figsize=(7, 3.6))
plt.hist(draws6, bins=range(6, 61), color="steelblue")
plt.axvline(mean6, color="red", ls="--", lw=2, label=f"mean = {mean6:.1f}")
plt.xlabel("Draws needed (n=6)")
plt.ylabel("Frequency")
plt.title("Distribution, not just the mean")
plt.legend()
plt.show()

print(f"평균        : {mean6:.1f}")
print(f"표준편차    : {draws6.std():.1f}")
# TODO 5: 15상자 안에 다 모을 확률을 구하세요.  힌트: (draws6 <= 15).mean()
print("15상자 안에 다 모을 확률:", None)

🤔 **평균은 약 14.9상자. 그런데 15상자를 사도 성공률은 63% 정도다.**

즉 **세 번에 한 번은 실패한다.** "90%는 확신하고 싶다"면 23상자가 필요하다 —
평균의 1.5배가 넘는다.

분포가 **오른쪽으로 길게 늘어져 있기** 때문이다(마지막 한 종류를 기다리는 시간이 길다).
평균은 이 긴 꼬리에 끌려가 올라가지만, 그 평균조차 "절반의 성공"을 보장하지 않는다.

> **"평균적으로 괜찮다"가 위험한 이유가 이것이다.**
> 서비스 응답시간, 배송 예정일, 프로젝트 일정 — 전부 같은 구조다.
> 실무에서 평균 대신 **95 퍼센타일**을 보는 이유이기도 하다.

---

## 마무리 — 자가 점검

- [ ] 문제를 조각($T_1, \ldots, T_n$)으로 쪼개 기댓값을 더할 수 있다
- [ ] 선형성이 독립을 요구하지 않는다는 것을 설명할 수 있다
- [ ] 시뮬레이션 값과 이론값 $nH_n$ 이 일치하는 것을 확인했다
- [ ] 평균과 분포가 다른 이야기를 한다는 것을 확인했다

**오늘 배운 것을 한 문장으로.**

> (여기에 작성)

### 📌 과제 2 — 게임의 기댓값 (`hw/W03_hw.md`), 4주차 수업 전까지